In [209]:
import matplotlib.pyplot as plt
import seaborn as sns
from utils import load_config
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler # used to normalize data for radar charts # conda install -c conda-forge scikit-learn   
from scipy.spatial.distance import euclidean
from fastdtw import fastdtw # pip install fastdtw

# To handle curvature issue 
# robot stagnate or slowly moves curvature explode
from scipy.signal import medfilt
from scipy.interpolate import interp1d

**Handling velocity**

In [210]:
def model_only_segments(df, teleop_col='lin_x_teleop'):
    """Return boolean mask and segment indices for contiguous model-controlled segments."""
    mask_model_only = df[teleop_col] == 0.0
    # label contiguous True regions
    # diff on mask: True->False transitions produce boundaries
    idx = np.where(mask_model_only)[0]
    if len(idx) == 0:
        print("No model segments found")
        return []  # no model segments
    # find breaks where indices are not consecutive
    breaks = np.where(np.diff(idx) != 1)[0] # if difference between consecutive indices is not 1, we have a jump in indices meaning we used teleoperation
    segments = []
    start = idx[0]
    for b in breaks:
        end = idx[b]
        segments.append((start, end))
        start = idx[b+1]
    segments.append((start, idx[-1]))
    return segments, breaks

## Process data

In [211]:
reference_header = 'reference'

config = load_config()
# TODO Handle now that we cannot be inside docker using plotly anymore

df = pd.read_csv(f"../dataframes/all_data_20251029-030058.csv") #Index(['pose_x', 'pose_y', 'goal', 'robot', 'environment', 'env_type','augmentation'],
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential


# Compute path lenght

In [212]:
# Take into account the whole path itself (with intervention)

def compute_path_length(df: pd.DataFrame):
    path_lengths = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        positions = group[['pose_x', 'pose_y']].to_numpy()
        length = 0.0
        for i in range(1, len(positions)):
            length += euclidean(positions[i-1], positions[i])
        path_lengths.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'path_length': length
        })
    return pd.DataFrame(path_lengths)


In [213]:
path_lengths_df = compute_path_length(df)
path_lengths_df.head()

,robot,environment,augmentation,env_type,path_length
0,bunker,mist_corridor,blur,indoor,7.654247
1,bunker,mist_corridor,no_augmentation,indoor,7.237963
2,bunker,mist_corridor,rain_torrential,indoor,6.290707
3,bunker,mist_corridor,sunFlare_physic,indoor,2.185343


In [214]:
df = df.merge(path_lengths_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707


# Compute number of intervention

In [215]:
# Failsafe as some bags don't have teleop data because there was no intervention at all
def intervention_count(df: pd.DataFrame, teleop_col: str = 'lin_x_teleop'):
    intervention_counts = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:

        if teleop_col not in group.columns:
            intervention_count = 0        
        else:
            _, breaks = model_only_segments(group, teleop_col=teleop_col)
            intervention_count = len(breaks) if len(breaks) > 0 else 0

        intervention_counts.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'intervention_count': intervention_count
        })
    return pd.DataFrame(intervention_counts)

In [216]:
intervention_counts_df = intervention_count(df, teleop_col='lin_x_teleop')
intervention_counts_df.head()

,robot,environment,augmentation,env_type,intervention_count
0,bunker,mist_corridor,blur,indoor,0
1,bunker,mist_corridor,no_augmentation,indoor,0
2,bunker,mist_corridor,rain_torrential,indoor,0
3,bunker,mist_corridor,sunFlare_physic,indoor,0


In [217]:
df = df.merge(intervention_counts_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length,intervention_count
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0


# Computing arrival rate based on CARE.  **(FOR NOW NOT A RATE)**

**Arrival rate** is defined as: *# goal reach w or w/o collisions* for our case its intervention

In [218]:
def count_arrival(df: pd.DataFrame):
    arrivals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        goal_reached = group['goal'].any()
        arrivals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'arrival': int(goal_reached)
        })
    return pd.DataFrame(arrivals)

In [219]:
count_arrival_df = count_arrival(df)
count_arrival_df.head()

,robot,environment,augmentation,env_type,arrival
0,bunker,mist_corridor,blur,indoor,1
1,bunker,mist_corridor,no_augmentation,indoor,1
2,bunker,mist_corridor,rain_torrential,indoor,1
3,bunker,mist_corridor,sunFlare_physic,indoor,1


In [220]:
df = df.merge(count_arrival_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length,intervention_count,arrival
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1


# Compute safe only arrivals

In [221]:
def count_arrival_safe(df: pd.DataFrame):
    arrivals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        goal_reached = group['goal'].any()

        teleop_used = group['lin_x_teleop'].any() if 'lin_x_teleop' in group.columns else False
        if teleop_used:
            goal_reached = False

        arrivals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'arrival_safe': int(goal_reached)
        })
    return pd.DataFrame(arrivals)

In [222]:
count_arrival_safe_df = count_arrival_safe(df)
count_arrival_safe_df.head()

,robot,environment,augmentation,env_type,arrival_safe
0,bunker,mist_corridor,blur,indoor,1
1,bunker,mist_corridor,no_augmentation,indoor,1
2,bunker,mist_corridor,rain_torrential,indoor,1
3,bunker,mist_corridor,sunFlare_physic,indoor,1


In [223]:
df = df.merge(count_arrival_safe_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length,intervention_count,arrival,arrival_safe
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1,1
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1,1
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1,1
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1,1
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0,1,1


___________________

# Count false goal positive 
**When model predict arrrived at goal but far from ground truth**

In [224]:
def compute_dist_to_goal(df: pd.DataFrame, reference_header: str = 'reference'):
    assert df["augmentation"].str.contains(reference_header).any(), f"Reference path not found in dataframe only {df['augmentation'].unique()}."
    dist_to_goals = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:
        # Get reference goal position
        df_reference = df[df['augmentation'] == reference_header]
        ref_group = df_reference[
            (df_reference['robot'] == robot) &
            (df_reference['environment'] == environment) &
            (df_reference['env_type'] == env_type)
        ]
        if ref_group.empty:
            print(f"No reference data for {robot}, {environment}, {env_type}")
            continue
        goal_pos = ref_group[['pose_x', 'pose_y']].iloc[-1].to_numpy()

        # Compute final distance to goal
        final_pos = group[['pose_x', 'pose_y']].iloc[-1].to_numpy()
        dist_to_goal = euclidean(final_pos, goal_pos)

        dist_to_goals.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'dist_to_goal': dist_to_goal
        })
    return pd.DataFrame(dist_to_goals)

In [225]:
def count_false_goal_positives(df: pd.DataFrame, goal_threshold: float = 0.5):
    false_positives = []
    grouped = df.groupby(['robot', 'environment', 'augmentation', 'env_type'])
    for (robot, environment, augmentation, env_type), group in grouped:

        
        if 'dist_to_goal' not in group.columns:
            dist_goal_df = compute_dist_to_goal(df)
            df = df.merge(dist_goal_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')


        goal_reached = group['goal'].any()
        final_distance = group['dist_to_goal'].iloc[-1]

        false_positive = goal_reached and (final_distance > goal_threshold)

        false_positives.append({
            'robot': robot,
            'environment': environment,
            'augmentation': augmentation,
            'env_type': env_type,
            'false_goal_positive': int(false_positive)
        })
    return pd.DataFrame(false_positives)

## Add refence dataframe

In [226]:
def load_reference_poses(df: pd.DataFrame, folder_path: str, reference_header: str = 'reference'):
    grouped = df.groupby(['robot', 'environment'])
    for (robot, environment), group in grouped:
        ref_dfs = pd.read_csv(f"{folder_path}/{robot}/{environment}/{reference_header}/{robot}_{environment}_{reference_header}_odom.csv")
        ref_dfs = ref_dfs[['pose.pose.position.x', 'pose.pose.position.y']]
        ref_dfs = ref_dfs.rename(columns={'pose.pose.position.x': 'pose_x', 'pose.pose.position.y': 'pose_y'})
        ref_dfs['robot'] = robot
        ref_dfs['environment'] = environment
        ref_dfs['augmentation'] = reference_header
        ref_dfs['env_type'] = group['env_type'].iloc[0] # iloc to retrieve only one value

    return ref_dfs

## Compute dist to goal

In [227]:
folder_path = "/home/mae/Documents/GIT/Research/SafeGNM/metrics/dataframes"
ref_dfs = load_reference_poses(df, folder_path)
# print(ref_dfs.head(10))

df = pd.concat([df, ref_dfs], ignore_index=True)
df.head()


,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length,intervention_count,arrival,arrival_safe
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0


In [228]:
df["augmentation"].unique()

array(['rain_torrential', 'sunFlare_physic', 'blur', 'no_augmentation',
       'reference'], dtype=object)

In [229]:
distance_to_goal_df = compute_dist_to_goal(df)
distance_to_goal_df.head()


,robot,environment,augmentation,env_type,dist_to_goal
0,bunker,mist_corridor,blur,indoor,1.034365
1,bunker,mist_corridor,no_augmentation,indoor,1.039398
2,bunker,mist_corridor,rain_torrential,indoor,0.643260
3,bunker,mist_corridor,reference,indoor,0.000000
4,bunker,mist_corridor,sunFlare_physic,indoor,3.446489


In [230]:
df = df.merge(distance_to_goal_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length,intervention_count,arrival,arrival_safe,dist_to_goal
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326


## Compute false goal positive

In [231]:
false_positive_df = count_false_goal_positives(df, goal_threshold=0.5)
false_positive_df.head()

,robot,environment,augmentation,env_type,false_goal_positive
0,bunker,mist_corridor,blur,indoor,1
1,bunker,mist_corridor,no_augmentation,indoor,1
2,bunker,mist_corridor,rain_torrential,indoor,1
3,bunker,mist_corridor,reference,indoor,0
4,bunker,mist_corridor,sunFlare_physic,indoor,1


In [232]:
df = df.merge(false_positive_df, on=['robot', 'environment', 'augmentation', 'env_type'], how='left')
df.head()

,pose_x,pose_y,Time,lin_x_model,lin_y_model,closest_node,goal,robot,environment,env_type,augmentation,path_length,intervention_count,arrival,arrival_safe,dist_to_goal,false_goal_positive
0,-0.006957,-0.001032,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326,1
1,-0.006472,-0.000402,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326,1
2,-0.006833,-0.000425,1.761696e+09,0.0,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326,1
3,-0.003967,-0.002378,1.761696e+09,0.1,0.0,NaN,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326,1
4,0.000525,-0.000635,1.761696e+09,0.1,0.0,3.0,False,bunker,mist_corridor,indoor,rain_torrential,6.290707,0.0,1.0,1.0,0.64326,1


___________________

# Path deviation

In [244]:
def fill_rows(current_df, target_length):

  current_len = len(current_df)

  if current_len < target_length:
      last_row = current_df.iloc[-1]
      # Compute how many rows to add
      n_to_add = target_length - current_len
      print(f"Filling {n_to_add} rows to reach target length {target_length} from current length {current_len}")

      # Create new rows using the last values
      new_rows = pd.DataFrame(
          [last_row] * n_to_add,
          columns=current_df.columns
      )

      print(f"df new rows {new_rows.head()}")
  return pd.concat([current_df, new_rows], ignore_index=True)



def compute_path_deviation(df, reference_header="reference"):

  path_deviation_list = []

  # print(f"aug are {df['augmentation'].unique()}")

  for (robot, environment, augmentation, env_type), group in df.groupby(['robot', 'environment', 'augmentation', 'env_type']):
    
    
    if augmentation != "sunFlare_physic" and augmentation != "no_augmentation":
      continue

    print(f"______________________________________________________________")

    print(f"Processing {group[["robot", "environment", "augmentation", "env_type"]].head(1)}")

    path_deviation = 0.0
    len_deviation = 0.0
    df_ref_deviation = pd.DataFrame()
    df_real_deviation = pd.DataFrame()

    df_ref = df[(df['robot'] == robot) & (df['environment'] == environment) & (df['augmentation'] == reference_header) & (df['env_type'] == env_type)]
    ref_poses = df_ref[['pose_x', 'pose_y']].copy()
    df_real = group.copy()
    real_poses = df_real[['pose_x', 'pose_y']].copy()

    # print(f"poses real {real_poses.head(1)}")
    # print(f"poses ref {ref_poses.head(1)}")

    # print(f"ref_poses length: {len(ref_poses)}")
    # print(f"real_poses length: {len(real_poses)}")

    # calculate the deviation between simulation and real
    diff_len = len(real_poses) - len(ref_poses)
    # print(f"diff len {diff_len}")
    # make the shorter df the same length as the larger one by repeating the last values
    if diff_len > 0: # this means that real is larger
      # print(f"Real is larger by {diff_len} steps for {robot}, {environment}, {augmentation}, {env_type}")
      df_ref_deviation = ref_poses.copy()
      # for _ in range(len(real_poses)):
      #   df_ref_deviation.loc[len(df_ref_deviation)] = {"pose_x": df_ref_deviation["pose_x"].iloc[-1], "pose_y": df_ref_deviation["pose_y"].iloc[-1]}
      df_ref_deviation = fill_rows(df_ref_deviation, len(real_poses))
      # print(f"start df_ref_deviation {df_ref_deviation.head(1)}")
      print(f"df_ref_deviation after fill {df_ref_deviation.tail()}")

      df_real_deviation = real_poses.copy()

    elif diff_len < 0: # this means that reference is larger
      # print(f"Reference is larger by {-diff_len} steps for {robot}, {environment}, {augmentation}, {env_type}")
      df_real_deviation = real_poses.copy()
      # for _ in range(len(ref_poses)):
      #   df_real_deviation.loc[len(df_real_deviation)] = {"pose_x": df_real_deviation["pose_x"].iloc[-1], "pose_y": df_real_deviation["pose_y"].iloc[-1]}
      df_real_deviation = fill_rows(df_real_deviation, len(ref_poses))
    
      df_ref_deviation = ref_poses.copy()

    else: # this means that both are the same length
      # print(f"Both reference and real are the same length for {robot}, {environment}, {augmentation}, {env_type}")
      df_ref_deviation = ref_poses.copy()
      df_real_deviation = real_poses.copy()


    assert len(df_ref_deviation) == len(df_real_deviation), f"Length mismatch after adjustment for {robot}, {environment}, {augmentation}, {env_type}: ref {len(df_ref_deviation)} vs actual {len(df_real_deviation)}"

    path_length_ref = np.sum(np.sqrt(df_ref['pose_x'].diff()**2 + df_ref['pose_y'].diff()**2)) # calculate the distances and then sum them


    path_deviation = np.sum(np.sqrt((df_ref_deviation["pose_x"] - df_real_deviation["pose_x"])**2
                                  + (df_ref_deviation["pose_y"] - df_real_deviation["pose_y"])**2)) / path_length_ref * 100
    
    
    # if augmentation == "no_augmentation" or augmentation == "sunFlare_physic":
    print(f"poses ref last {df_ref_deviation["pose_x"].to_numpy()[-1]} before last {df_ref_deviation["pose_x"].to_numpy()[-2]}")
    print(f"poses real last {df_real_deviation["pose_x"].to_numpy()[-1]} before last {df_real_deviation["pose_x"].to_numpy()[-2]}")

    print("- - - - - - - - - - ")

    print(f"{df_ref_deviation["pose_x"] - df_real_deviation["pose_x"]}")
    print(f"{df_ref_deviation["pose_y"] - df_real_deviation["pose_y"]}")

    print("- - - - - - - - - - ")


    print(f"path dev {np.sum(np.sqrt((df_ref_deviation["pose_x"] - df_real_deviation["pose_x"])**2
                                  + (df_ref_deviation["pose_y"] - df_real_deviation["pose_y"])**2))}")
    
    # print(f"path dev normalize {np.sum(np.sqrt((df_ref_deviation["pose_x"] - df_real_deviation["pose_x"])**2
    #                               + (df_ref_deviation["pose_y"] - df_real_deviation["pose_y"])**2)) / path_length_ref}")
    
    # print(f"path dev normalize 100 {np.sum(np.sqrt((df_ref_deviation["pose_x"] - df_real_deviation["pose_x"])**2
    #                               + (df_ref_deviation["pose_y"] - df_real_deviation["pose_y"])**2)) / path_length_ref} * 100")


    print(f"______________________________________________________________")

    len_deviation = diff_len

    path_deviation_list.append({
        'robot': robot,
        'environment': environment,
        'augmentation': augmentation,
        'env_type': env_type,
        'path_deviation': path_deviation,
        'len_deviation': len_deviation
      })
  
  return pd.DataFrame(path_deviation_list)

In [245]:
df_deviation = compute_path_deviation(df, reference_header='reference')
# df_deviation.head()

______________________________________________________________
Processing        robot    environment     augmentation env_type
1046  bunker  mist_corridor  no_augmentation   indoor
Filling 414 rows to reach target length 460 from current length 46
df new rows         pose_x    pose_y
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
df_ref_deviation after fill        pose_x    pose_y
455  5.239502  0.350758
456  5.239502  0.350758
457  5.239502  0.350758
458  5.239502  0.350758
459  5.239502  0.350758
poses ref last 5.239502429962158 before last 5.239502429962158
poses real last 6.278899669647217 before last 6.282130241394043
- - - - - - - - - - 
0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
1501   NaN
1502   NaN
1503   NaN
1504   NaN
1505   NaN
Name: pose_x, Length: 920, dtype: float64
0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
1501   NaN
1502   NaN
1503   NaN
1504   

In [235]:
df_deviation = compute_path_deviation(df, reference_header='reference')
df_deviation.head()
# df_deviation[df_deviation["augmentation"] == "reference"].head()

Filling 414 rows to reach target length 460 from current length 46
df new rows         pose_x    pose_y
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
poses ref last 5.239502429962158 before last 5.239502429962158
poses real last 6.278899669647217 before last 6.282130241394043
Filling 101 rows to reach target length 147 from current length 46
df new rows         pose_x    pose_y
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
1551  5.239502  0.350758
poses ref last 5.239502429962158 before last 5.239502429962158
poses real last 1.798036813735962 before last 1.8009707927703855


,robot,environment,augmentation,env_type,path_deviation,len_deviation
0,bunker,mist_corridor,no_augmentation,indoor,0.0,414
1,bunker,mist_corridor,sunFlare_physic,indoor,0.0,101
